# 07 · Adım 8B-1 — şehir-bırakmalı katlamalar + bina bileşeni olasılık dökümü
**Ön-kayıt:** `docs/adim8b_onkayit.md` §3 (8B-1), 30 Ekim 2026. Yeni mimari YOK; ChangeOS, ön-kayıtlı varsayılanlar, seed 42, 50 epoch (tam eğitimle aynı). **Test bölmesine DOKUNMAZ** (Antakya/e6/e7 açılmaz; kod `--allow-test` olmadan zaten kilitli).

**Üç iş (sırayla):**
1. **Osmaniye-dışarıda (VAR):** yeni eğitim YOK; 05 not defterinin `trigger_osmaniye_seed42/model_final.pt` ağırlığıyla yalnız değerlendirme + olasılık dökümü.
2. **Adıyaman-dışarıda (YENİ)** ve 3. **Malatya-dışarıda (YENİ):** iki GPU'da paralel eğitim (her biri ≈1,7–2,1 sa tahmini) + dışarıdaki şehirde değerlendirme + döküm.

**Kaggle ayarları:** GPU **T4 x2** · Internet **On** · Secrets: `GITHUB_TOKEN` **Attached** · Input: dataset `melihkotman/disasterm3-turkiye` **ve** 05 tam eğitim not defterinin Output'u (Osmaniye ağırlığı için).

**Kullanım:** `MODE="dry"` (≈10–15 dk, kilit + döküm yolu) → sorun yoksa `MODE="run"`; ikisi de *Save & Run All (Commit)*.

**Dürüstlük notu:** Malatya katlaması pilotta ayarlama katlamasıydı (oylama `sqrt_inv` orada seçildi). Burada ham (oylamasız) olasılıklar kaydedilir; yine de Malatya sonuçları tamamen "dokunulmamış" değildir → sınırlılık.

In [ ]:
# ---- PARAMETRELER ----
MODE = "dry"                  # "dry" | "run"
CODE_REF = "adim8b-1-v1"      # yerelde: git tag adim8b-1-v1 && git push origin adim8b-1-v1
EPOCHS = 50
SEED = 42
FOLDS = {"A": "Adiyaman", "B": "Malatya"}     # şerit -> dışarıdaki şehir (GPU 0 / GPU 1)
print("MODE:", MODE, "| CODE_REF:", CODE_REF, "| FOLDS:", FOLDS)

In [ ]:
# 1) Yardımcılar + kurulum (ever-beta 0.5.6 sabit!)
import os, sys, json, time, glob, hashlib, subprocess, threading, shutil

def sh(cmd, check=True, mask=(), tail=3000):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    for m in mask:
        out = out.replace(m, "***")
    if out.strip():
        print(out.strip()[-tail:])
    if check and r.returncode:
        raise SystemExit(f"komut hata verdi (çıkış {r.returncode}); çıktıya bak")
    return r.returncode

sh('pip install -q timm "ever-beta==0.5.6" albumentations tifffile einops "datasets[vision]"')
sh("pip install -q --no-deps git+https://github.com/Z-Zheng/pytorch-change-models.git")
sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv")


In [ ]:
# 2) Kodu al, etikete sabitle, sürümü kaydet
REPO = "/kaggle/working/deprem-vlm-rag"

def get_code():
    if os.path.exists(REPO):
        return
    tok = None
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as e:
        print("GITHUB_TOKEN okunamadı:", type(e).__name__)
    if tok:
        rc = sh(f"git clone -q https://{tok}@github.com/MelihKotman/Deprem-VLM-RAG.git {REPO}", check=False, mask=(tok,))
        if rc == 0:
            sh(f"git -C {REPO} remote set-url origin https://github.com/MelihKotman/Deprem-VLM-RAG.git")
    if not os.path.exists(REPO):
        cand = glob.glob("/kaggle/input/**/deprem-vlm-rag", recursive=True)
        if not cand:
            raise SystemExit("kod alınamadı: GITHUB_TOKEN 'Attached' mı? ya da 'kod' dataset'ini ekle")
        shutil.copytree(cand[0], REPO)
        print("UYARI: kod 'kod' dataset'inden alındı (git etiketi doğrulanamaz)")

get_code()
git_sha = None
if os.path.exists(f"{REPO}/.git"):
    sh(f"git -C {REPO} fetch -q --tags", check=False)
    if sh(f"git -C {REPO} checkout -q {CODE_REF}", check=False):
        raise SystemExit(f"CODE_REF '{CODE_REF}' bulunamadı: yerelde `git tag {CODE_REF}` + `git push origin {CODE_REF}` yap")
    git_sha = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
files = sorted(glob.glob(f"{REPO}/src/*.py")) + [f"{REPO}/config/base.yaml"]
src_sha1 = hashlib.sha1(b"".join(open(f, "rb").read() for f in files)).hexdigest()
pipv = subprocess.run("pip freeze 2>/dev/null | grep -i -E '^(torch|torchvision|ever-beta|torchange|timm|numpy|opencv)'", shell=True, capture_output=True, text=True).stdout
with open("/kaggle/working/code_version.txt", "w") as f:
    f.write(f"code_ref={CODE_REF}\ngit_sha={git_sha}\nsrc_config_sha1={src_sha1}\n{pipv}")
print(open("/kaggle/working/code_version.txt").read())
os.chdir(REPO); sys.path.insert(0, REPO)
sh("python -m src.config && python -m src.dm3_verify", tail=1500)
sh("python -m src.selftest_eval", tail=800)


In [ ]:
# 3) Koşu yardımcıları (alt süreç + günlük dosyası; ekrana yalnız ilerleme/hata satırları)
RUNS = "/kaggle/working/runs"; LOGS = "/kaggle/working/logs"
os.makedirs(RUNS, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
CACHE_ROOT = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"   # çıktıya (Output) girmesin
print("önbellek kökü:", CACHE_ROOT); sh(f"df -h {CACHE_ROOT} | tail -1")
SHOW = ("İLERLEME", "Traceback", "Error", "error", "timing:", "tetikleyici:", "metrikler", "out of memory", "EXTRAPOLATION", "[S1]", "[S2]", "--max-minutes", "kilitli", "ayrılmıştır")

def run_job(tag, args, gpu, cache, workers, module="src.train_changeos"):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    cmd = [sys.executable, "-m", module, *args, "--cache-dir", cache, "--workers", str(workers)]
    oom = False
    with open(f"{LOGS}/{tag}.log", "a") as log:
        p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line)
            if "out of memory" in line.lower():
                oom = True
            if any(k in line for k in SHOW):
                print(f"[{tag}] {line.rstrip()[:260]}", flush=True)
        p.wait()
    return p.returncode, oom

def train_fold(city, gpu, cache, workers, epochs=EPOCHS, pilot=False):
    name = f"fold_{city.lower()}_seed{SEED}" + ("_dry" if pilot else "")
    out = f"{RUNS}/{name}"
    if os.path.exists(f"{out}/model_final.pt") and os.path.exists(f"{out}/components.csv.gz"):
        print(f"[{name}] zaten tamam, atlanıyor"); return 0
    base = ["--scenario", "S1", "--holdout", city, "--epochs", str(epochs), "--seed", str(SEED), "--max-minutes", "600",
            "--dump-components", "--out", out]
    if pilot:
        base += ["--pilot", "--pilot-tiles", "64"]
    extra = []
    for attempt in range(3):
        args = base + extra + (["--resume"] if os.path.exists(f"{out}/last.pt") else [])
        rc, oom = run_job(name, args, gpu, cache, workers)
        if rc == 0: return 0
        if rc == 3: continue
        if oom and "--no-cudnn-benchmark" not in extra:
            extra = ["--no-cudnn-benchmark"]; print(f"[{name}] OOM → --no-cudnn-benchmark"); continue
        break
    print(f"[{name}] BAŞARISIZ (çıkış {rc}); günlük: {LOGS}/{name}.log")
    return rc


In [ ]:
# 4) Osmaniye-dışarıda: MEVCUT tetikleyici ağırlığıyla yalnız değerlendirme + olasılık dökümü (yeni eğitim yok)
cand = sorted(glob.glob("/kaggle/input/**/trigger_osmaniye_seed42/model_final.pt", recursive=True))
print("Osmaniye ağırlığı adayları:", cand)
OSM_RC = None
if not cand:
    print("UYARI: 05 not defteri Output'u Input olarak ekli değil → bu adım atlandı. Ekle ve yeniden çalıştır (dry/run fark etmez).")
else:
    out = f"{RUNS}/fold_osmaniye_seed42_eval"
    if os.path.exists(f"{out}/components.csv.gz"):
        print("zaten tamam, atlanıyor"); OSM_RC = 0
    else:
        args = ["--scenario", "S1", "--eval-only", "--holdout", "Osmaniye", "--trigger-fold", "--ckpt", cand[0],
                "--dump-components", "--out", out]
        if MODE == "dry": args += ["--eval-max-tiles", "6"]
        OSM_RC, _ = run_job("osmaniye_eval", args, 0, f"{CACHE_ROOT}/cache_A", 4)
    print("Osmaniye değerlendirme çıkışı:", OSM_RC)
    if OSM_RC == 0 and MODE == "run":
        m = json.load(open(f"{out}/metrics.json"))["by"]["ALL"]
        print("KONTROL (tetikleyici metrics.json ile aynı olmalı: katı hasar-F1 0,071, macro 0,344):",
              round(m["strict"]["damage_f1"], 3), round(m["strict"]["macro_f1"], 3))

In [ ]:
# 5) DRY: iki fold'un 2 epoch'luk pilot hattı (döküm yazılıyor mu?) + Osmaniye kilidi; RUN: tam eğitim
t0 = time.time()
if MODE == "dry":
    res = {}
    def dry(l):
        res[l] = train_fold(FOLDS[l], "AB".index(l), f"{CACHE_ROOT}/cache_{l}", 2, epochs=2, pilot=True)
    th = [threading.Thread(target=dry, args=(l,)) for l in FOLDS]
    [t.start() for t in th]; [t.join() for t in th]
    print("DRY çıkış kodları:", res, "| süre dk:", round((time.time() - t0) / 60, 1))
else:
    res = {}
    def lane(l):
        res[l] = train_fold(FOLDS[l], "AB".index(l), f"{CACHE_ROOT}/cache_{l}", 2)
    th = [threading.Thread(target=lane, args=(l,)) for l in FOLDS]
    [t.start() for t in th]; [t.join() for t in th]
    print("ÇIKIŞ KODLARI:", res, "| TOPLAM duvar süresi sa:", round((time.time() - t0) / 3600, 2))

In [ ]:
# 6) Özet: döküm dosyaları + ilk bakış (YORUM YOK; kapı analizi ayrı sohbet/kod). Test'e dokunulmadı.
import pandas as pd
for d in sorted(glob.glob(f"{RUNS}/fold_*/")):
    p = d + "components.csv.gz"
    if not os.path.exists(p): print(d, "→ components.csv.gz YOK"); continue
    c = pd.read_csv(p); g = c[c.kind == "gt"]; f = c[c.kind == "fp"]
    pdm = (g.m2 + g.m3) / (g.m1 + g.m2 + g.m3).clip(lower=1e-9)
    print(f"{os.path.basename(d.rstrip('/'))}: gt bileşen {len(g)} (sınıf 1/2/3 = {[(g.gt_cls==k).sum() for k in (1,2,3)]}), fp {len(f)}, "
          f"yakalanamayan(loc_cover<0.5) {(g.loc_cover<0.5).sum()}, ikili hasar olasılığı ort. (gt sağlam/hasarlı∪yıkık) "
          f"{pdm[g.gt_cls==1].mean():.3f} / {pdm[g.gt_cls>=2].mean():.3f}")
for d in glob.glob(f"{RUNS}/*/"):
    if os.path.exists(d + "model_final.pt") and os.path.exists(d + "last.pt"):
        os.remove(d + "last.pt")
sh("du -sh /kaggle/working/runs /kaggle/working/logs | cat")
print("Bitti. Çıktıyı (components.csv.gz, tile_scores.csv, metrics.json) bilgisayara indirip RAGEarthQuake/ciktilar/adim8b1/ altına koy.")